# 7장. MOMENT 기초 모델과 모델 크기

In [1]:
# [준비 셀] 모든 노트북의 첫 코드 셀입니다. 항상 먼저 실행하세요.
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parents[1]          # docs/manual → 프로젝트 루트
assert (ROOT / "src/bearing_kd").exists(), "docs/manual 폴더에서 노트북을 열어야 합니다"
CODE = ROOT / "docs/manual/code"                 # 챕터별 완성 스크립트 위치
sys.path.insert(0, str(CODE))                    # 스크립트의 함수를 import 해서 쓰기 위해
print("프로젝트 루트 확인 완료:", ROOT.name)

프로젝트 루트 확인 완료: vib-onnx


진동 신호의 특징을 처음부터 스스로 배우는 모델을 만들려면 데이터가 아주 많이 필요합니다. 그래서 이 프로젝트는 **이미 많은 시계열로 학습된 모델(foundation model)**을 가져와 씁니다. 그 모델이 **MOMENT**입니다.

- 만든 곳: Carnegie Mellon University AutonLab
- 허깅페이스: [AutonLab/MOMENT-1-base](https://huggingface.co/AutonLab/MOMENT-1-base), [AutonLab/MOMENT-1-small](https://huggingface.co/AutonLab/MOMENT-1-small)
- 날씨, 전력, 심전도, 교통 등 다양한 시계열로 사전학습했습니다.

비유하면, 파형을 읽는 눈은 이미 훈련된 사람에게 "정상/고장 판정"만 새로 가르치는 것입니다.

## 7.1 모델 받기

```bash
python scripts/download_models.py      # models/MOMENT-1-base, models/MOMENT-1-small 에 저장
```

In [2]:
import os, warnings
os.environ["HF_HUB_OFFLINE"] = "1"                 # 인터넷 없이 로컬 모델 폴더만 사용
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"    # Mac: MPS가 모르는 연산은 CPU로
warnings.filterwarnings("ignore")
import torch
from ch09_train_teacher import pick_device
DEVICE = pick_device()
print("장치:", DEVICE)

장치: mps


In [3]:
for size in ("base", "small"):
    d = ROOT / f"models/MOMENT-1-{size}"
    files = {p.name: f"{p.stat().st_size / 1e6:.0f} MB" for p in d.iterdir() if p.is_file() and p.suffix in (".json", ".safetensors")}
    print(f"MOMENT-1-{size}:", files)

MOMENT-1-base: {'model.safetensors': '454 MB', 'config.json': '0 MB'}
MOMENT-1-small: {'model.safetensors': '152 MB', 'config.json': '0 MB'}


## 7.2 MOMENT 안에서 일어나는 일

512개 샘플 청크 하나가 들어가서 점수 2개가 나오기까지 다섯 단계를 거칩니다.

```text
청크 [1, 512] + mask [512]
  │
  ├─ ① normalizer (RevIN)      청크마다 평균 0, 표준편차 1로 맞춘다. mask=0인 칸은 계산에서 뺀다
  ├─ ② tokenizer (patching)    512개를 8개씩 묶어 패치 64개로 만든다       → [64, 8]
  ├─ ③ patch_embedding          패치 하나(8개 숫자)를 d_model개 숫자로 늘린다 → [64, d_model]
  ├─ ④ encoder (T5 Transformer) 패치 64개가 서로를 참고하며(attention) 특징을 다듬는다 (몸통)
  └─ ⑤ head                     패치 64개의 특징을 평균 → 선형층 → [정상 점수, 고장 점수]
```

- ①은 크기(진폭) 차이를 없앱니다. 그래서 따로 정규화할 필요가 없고, 대신 **진폭 정보는 모델에 전달되지 않습니다.**
- ④가 모델의 대부분(99.9%)이고, 사전학습으로 이미 똑똑합니다. 우리는 이 부분을 **고정(freeze)**합니다.
- ⑤ head만 새로 만들어 학습합니다. 사전학습 모델에는 분류용 head가 없기 때문입니다.

In [4]:
import ch07_moment as ch07

small = ch07.load_moment(ROOT / "models/MOMENT-1-small").eval()
print([name for name, _ in small.named_children()])
print(small.head)

Loading weights from local directory
['normalizer', 'tokenizer', 'patch_embedding', 'encoder', 'head']
ClassificationHead(
  (dropout): Dropout(p=0.1, inplace=False)
  (linear): Linear(in_features=512, out_features=2, bias=True)
)


## 7.3 Base vs Small: 모델 크기

`d_model`은 패치 하나를 표현하는 숫자 개수입니다. 층과 d_model이 클수록 표현력이 좋지만 무겁고 느립니다.

In [5]:
print("■ Small"); ch07.describe(small)
base = ch07.load_moment(ROOT / "models/MOMENT-1-base").eval()
print("\n■ Base"); ch07.describe(base)

■ Small
d_model(특징 길이) 512, 층 8개, attention head 6개, 패치 길이 8, 입력 길이 512
  patch_embedding         4,608
  encoder            35,332,800
  head                    1,026
  합계                 35,338,434  (float32 기준 141 MB)
  그중 학습 가능         1,026  ← 머리(head)뿐
  그중 안 쓰는 글자 임베딩 16,449,536개 (ONNX에서는 빠진다, 14장)


Loading weights from local directory

■ Base
d_model(특징 길이) 768, 층 12개, attention head 12개, 패치 길이 8, 입력 길이 512
  patch_embedding         6,912
  encoder           109,628,544
  head                    1,538
  합계                109,636,994  (float32 기준 439 MB)
  그중 학습 가능         1,538  ← 머리(head)뿐
  그중 안 쓰는 글자 임베딩 24,674,304개 (ONNX에서는 빠진다, 14장)


| | MOMENT-1-base (Teacher) | MOMENT-1-small (Student) |
|---|---|---|
| d_model | 768 | 512 |
| Transformer 층 | 12 | 8 |
| attention head | 12 | 6 |
| 전체 파라미터 | 약 1억 1천만 | 약 3천5백만 |
| 파일 | 454 MB | 152 MB |
| head (512→2, 768→2) | 1,538 | 1,026 |

> **"글자용 임베딩"**: MOMENT의 몸통은 원래 문장용 모델(T5)이라 글자 사전(32,128 × d_model)이 들어 있습니다. 시계열에서는 쓰지 않아서 계산에 참여하지 않고, ONNX로 바꾸면 빠집니다(14장). 그래서 Small 전체는 141 MB인데 ONNX 가중치 파일은 86 MB입니다.

## 7.4 "몸통 고정"의 의미

`requires_grad`는 "이 숫자를 학습 중에 고쳐도 되는가"입니다. `freeze_encoder=True`로 불러오면 몸통은 모두 `False`, head만 `True`입니다.

In [6]:
for name, p in list(small.named_parameters())[:3] + list(small.named_parameters())[-2:]:
    print(f"{name:60s} {str(tuple(p.shape)):12s} 학습={p.requires_grad}")

patch_embedding.mask_embedding                               (512,)       학습=False
patch_embedding.value_embedding.weight                       (512, 8)     학습=False
encoder.embed_tokens.weight                                  (32128, 512) 학습=False
head.linear.weight                                           (2, 512)     학습=True
head.linear.bias                                             (2,)         학습=True


## 7.5 실제 진동 청크를 넣어 보기

6장의 DataLoader에서 윈도우 2개를 꺼내 청크 16개로 만든 뒤 Small에 넣습니다.

In [7]:
import json
from torch.utils.data import DataLoader
from ch06_dataset import WindowDataset, make_moment_chunks

split = json.loads((ROOT / "outputs/manual/split_holdout.json").read_text())
ds = WindowDataset([ROOT / f for f in split["train"]])
x, y = next(iter(DataLoader(ds, batch_size=2)))
chunks, mask, _ = make_moment_chunks(x)
with torch.no_grad():
    out = small(x_enc=chunks, input_mask=mask)
print("입력 chunks", tuple(chunks.shape), "→ logits", tuple(out.logits.shape))
print("윈도우 1의 청크 8개 logits:\n", out.logits[:8].numpy().round(3))
print("\n아직 head가 학습 전이라 정상/고장과 상관없는 값입니다. 9장에서 학습합니다.")

입력 chunks (16, 1, 512) → logits (16, 2)
윈도우 1의 청크 8개 logits:
 [[ 0.468 -0.369]
 [ 0.469 -0.35 ]
 [ 0.461 -0.345]
 [ 0.446 -0.391]
 [ 0.463 -0.384]
 [ 0.46  -0.391]
 [ 0.44  -0.407]
 [ 0.463 -0.379]]

아직 head가 학습 전이라 정상/고장과 상관없는 값입니다. 9장에서 학습합니다.


## 7.6 불러오기 설정 정리

```python
MOMENTPipeline.from_pretrained(model_dir, local_files_only=True, model_kwargs={
    "task_name": "classification",   # 분류 head를 붙인다
    "n_channels": 1,                 # 진동 1채널
    "num_class": 2,                  # 정상/고장
    "freeze_encoder": True,          # 몸통 고정
    "freeze_embedder": True,         # 패치 임베딩 고정
    "enable_gradient_checkpointing": False,  # 8장에서 설명
})
model.init()                         # head를 무작위 값으로 새로 만든다
```

`model.init()` 직전에 `torch.manual_seed(42)`로 무작위 시작점을 고정해야 head 초기값이 매번 같습니다(재현성).

## 7장 완성 코드: `code/ch07_moment.py`

In [8]:
%run code/ch07_moment.py --size small

Loading weights from local directory
MOMENT-1-small  (파일 152 MB)
d_model(특징 길이) 512, 층 8개, attention head 6개, 패치 길이 8, 입력 길이 512
  patch_embedding         4,608
  encoder            35,332,800
  head                    1,026
  합계                 35,338,434  (float32 기준 141 MB)
  그중 학습 가능         1,026  ← 머리(head)뿐
  그중 안 쓰는 글자 임베딩 16,449,536개 (ONNX에서는 빠진다, 14장)

입력 (3, 1, 512) → logits (3, 2)
logits: [[0.476, -0.339], [0.465, -0.329], [0.416, -0.304]]  ← 머리가 아직 학습 전이라 의미 없는 값


In [9]:
from IPython.display import Code
Code(filename=str(CODE / "ch07_moment.py"), language="python")

"""7장 완성 코드: MOMENT 사전학습 모델을 불러와 구조와 크기를 확인하고, 진동 청크를 넣어 본다.

실행 (프로젝트 루트에서):
    python docs/manual/code/ch07_moment.py                      # Small
    python docs/manual/code/ch07_moment.py --size base          # Base
"""

import argparse
import os
import warnings
from pathlib import Path

import torch

ROOT = Path(__file__).resolve().parents[3]
os.environ.setdefault("HF_HUB_OFFLINE", "1")   # 인터넷 없이 로컬 모델 폴더만 사용


def load_moment(model_dir, seed=42):
    """분류용 MOMENT를 만들고, 사전학습 가중치(몸통)는 모두 고정한다."""
    from momentfm import MOMENTPipeline

    # MOMENT는 "분류 head는 사전학습되지 않았다"는 경고를 낸다. 우리가 head를 학습하므로 정상이라 숨긴다.
    warnings.filterwarnings("ignore", message="Only reconstruction head is pre-trained")
    model = MOMENTPipeline.from_pretrained(
        str(model_dir), local_files_only=True,
        model_kwargs={
            "task_name": "classification",   # 분류용 머리(head)를 붙인다
            "n_channels": 1,                 # 진동 1채널
            "num_class": 2,                  # 정상 / 고장
            "freeze_encoder": True,          # 인코더(몸통) 고정
            "freeze_embedder": True,         # 패치 임베딩 고정
            "enable_gradient_checkpointing": False,  # 8장에서 설명
        },
    )
    torch.manual_seed(seed)                  # head 초기값을 매번 같게 (재현성)
    model.init()                             # 머리(head)를 새로 만든다 (무작위 초기값)
    return model


def count_params(model):
    """부분별 파라미터(학습 가능한 숫자) 개수."""
    parts = {}
    for name, p in model.named_parameters():
        key = name.split(".")[0]
        parts[key] = parts.get(key, 0) + p.numel()
    total = sum(parts.values())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    unused = model.encoder.embed_tokens.weight.numel()   # 글자용 임베딩: 시계열에서는 안 쓰임
    return parts, total, trainable, unused


def describe(model):
    parts, total, trainable, unused = count_params(model)
    cfg = model.encoder.config
    print(f"d_model(특징 길이) {model.config.d_model}, 층 {len(model.encoder.block)}개, "
          f"attention head {cfg.num_heads}개, 패치 길이 {model.patch_len}, 입력 길이 {model.seq_len}")
    for k, v in parts.items():
        print(f"  {k:16s} {v:>12,}")
    print(f"  {'합계':16s} {total:>12,}  (float32 기준 {total * 4 / 1e6:.0f} MB)")
    print(f"  그중 학습 가능  {trainable:>12,}  ← 머리(head)뿐")
    print(f"  그중 안 쓰는 글자 임베딩 {unused:,}개 (ONNX에서는 빠진다, 14장)")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--size", choices=("small", "base"), default="small")
    args = parser.parse_args()
    model_dir = ROOT / f"models/MOMENT-1-{args.size}"
    model = load_moment(model_dir).eval()
    print(f"MOMENT-1-{args.size}  (파일 {os.path.getsize(model_dir / 'model.safetensors') / 1e6:.0f} MB)")
    describe(model)

    torch.manual_seed(0)
    x = torch.randn(3, 1, 512)                   # 청크 3개 [batch, 채널, 512]
    mask = torch.ones(3, 512)
    with torch.no_grad():
        out = model(x_enc=x, input_mask=mask)
    print(f"\n입력 {tuple(x.shape)} → logits {tuple(out.logits.shape)}")
    print("logits:", [[round(float(v), 3) for v in row] for row in out.logits], " ← 머리가 아직 학습 전이라 의미 없는 값")


if __name__ == "__main__":
    main()